# Caracal s07 - Bench cybersec em TPU v3-8

**Settings**: Accelerator -> TPU VM v3-8 · Internet ON

O problema do run TPU anterior foi `.generate()` recompilando o grafo XLA a
cada shape nova de prompt. Aqui `IGNITE_XLA=1` liga padding pra bucket fixo
(512/1024/2048/4096) + `min_new_tokens == max_new_tokens`, entao o numero de
shapes distintos fica pequeno e a compilacao amortiza.

**Aviso honesto**: isso ainda nao foi validado em TPU. As primeiras questoes
vao ser lentas (compilacao por bucket). Se depois de ~20 questoes o ritmo nao
estabilizar, o caminho TPU nao vale a pena pra esse stack - mate e volte pra GPU.
A celula de smoke test abaixo mede exatamente isso antes de gastar as 20h.

In [ ]:
import os
BASE_MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct'
CHECKPOINT_DATASET = 'arturpn/caracal-base-3b-s04'
# Sufixo por run: dois kernels publicando no mesmo id fazem o segundo
# sobrescrever o primeiro (cada um so tem o proprio arquivo na pasta).
RUN_TAG = os.environ.get('RUN_TAG', 'adapter')
OUTPUT_DATASET = f'pedroafonso2/caracal-bench-{RUN_TAG}'
# cti_bench primeiro: RCM (CVE->CWE) e a claim que compara direto vs Foundation-Sec-8B.
# Depois MCQ curtos, que sofrem menos com o padding do XLA.
BENCHES = ['cti_bench', 'cybermetric', 'secqa', 'secbench', 'mmlu_security', 'seceval', 'cybersoceval']
print(f'{BASE_MODEL} + {CHECKPOINT_DATASET} -> TPU')

In [ ]:
!pip install -q -U 'transformers==4.49.0' 'peft==0.14.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'sentence-transformers' 'scipy' 'statsmodels' 'antlr4-python3-runtime==4.11' kaggle huggingface_hub
import torch, torch_xla
import torch_xla.core.xla_model as xm
print('torch', torch.__version__, '| xla device:', xm.xla_device())

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
os.environ['IGNITE_XLA'] = '1'   # liga padding de shape fixo
os.environ['PJRT_DEVICE'] = 'TPU'
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())

In [ ]:
import subprocess
ckpt_dir = '/kaggle/working/ckpt'
subprocess.run(['kaggle', 'datasets', 'download', '-d', CHECKPOINT_DATASET,
                '-p', ckpt_dir, '--unzip', '--force'], check=True)
subprocess.run(['ls', '-la', ckpt_dir], check=True)

In [ ]:
import torch, time
import torch_xla.core.xla_model as xm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

dev = xm.xla_device()
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'   # decoder-only: pad a esquerda ou a geracao sai torta

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(model, ckpt_dir)
model = model.to(dev).eval()
print(f'modelo na TPU em {time.time()-t0:.0f}s')

In [ ]:
# GATE DURO: mede se o custo por questao estabiliza depois da compilacao XLA.
# Evidencia do run de 2026-07-19: q1 levou 9538s (2h39) de compilacao e a media
# quente ficou em 21.6s/questao -> 500q x 7 benches nao cabe. O gate ANTES so
# imprimia aviso e seguia, queimando o run inteiro ate DeadKernel. Agora aborta.
import time, sys
from eval.s07.benches._common import generate

WARM_MAX_S = 10.0   # acima disso o caminho TPU nao paga

probe = 'Question: What does CIA stand for in security?\n\nA. x\nB. y\nC. z\nD. w\n\nAnswer with A, B, C, or D inside \\boxed{}.'
chat = tok.apply_chat_template([{'role':'user','content':probe}], tokenize=False, add_generation_prompt=True)
times = []
for i in range(10):
    t = time.time(); generate(model, tok, chat, max_new=32); dt = time.time()-t
    times.append(dt); print(f'  q{i+1}: {dt:.1f}s', flush=True)

warm = sum(times[5:]) / 5
print(f'\nprimeira: {times[0]:.1f}s | media das ultimas 5: {warm:.1f}s')
if warm > WARM_MAX_S:
    raise SystemExit(
        f'ABORTADO: XLA nao estabilizou ({warm:.1f}s/questao > {WARM_MAX_S}s). '
        f'500 questoes = {warm*500/3600:.1f}h por bench. Rode na GPU.')
print(f'>> OK. Estimativa pra 500 questoes: {warm*500/60:.0f} min por bench.')


In [ ]:
import json, os
from eval.s07.benches import BENCH_REGISTRY

os.makedirs('/kaggle/working/bench-out', exist_ok=True)
# kwargs por bench (explicito, sem if/elif fragil). cti_bench: RCM 500 + MCQ 500
# em vez dos 1000/2500 completos, pra caber nas 20h se o XLA estiver lento.
KW = {
    'cti_bench':    {'n_rcm': 500, 'n_mcq': 500, 'subsets': ['rcm', 'mcq']},
    'cybermetric':  {'tier': 500},
    'secqa':        {},
    'secbench':     {'n_mcq': 300},
    'mmlu_security':{'n': 100},
    'seceval':      {'n': 200},
    'cybersoceval': {'n': 100},
}
results = {'model': BASE_MODEL, 'adapter': ckpt_dir, 'device': 'tpu-v3-8'}
for name in BENCHES:
    if name not in BENCH_REGISTRY:
        continue
    print(f'\n=== {name} ===', flush=True)
    try:
        results[name] = BENCH_REGISTRY[name](model, tok, **KW.get(name, {'n': 100}))
    except Exception as e:
        results[name] = {'error': f'{type(e).__name__}: {e}'}
        print('  falhou:', str(e)[:200])
    with open('/kaggle/working/bench-out/s07-tpu-eval.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)   # salva a cada bench
print('\nfim')

In [ ]:
import json
res = json.load(open('/kaggle/working/bench-out/s07-tpu-eval.json'))
print('=== TPU ===')
for k, v in res.items():
    if not isinstance(v, dict):
        continue
    if 'error' in v:
        print(f"{k:20s}: ERRO {v['error'][:70]}")
    elif 'accuracy' in v:
        print(f"{k:20s}: {v['accuracy']*100:5.1f}%  n={v.get('n',0)}")
    else:
        for sub, sv in v.items():
            if isinstance(sv, dict) and 'accuracy' in sv:
                print(f"{k}.{sub:14s}: {sv['accuracy']*100:5.1f}%  n={sv.get('n',0)}")

In [ ]:
import json, subprocess
from pathlib import Path
pub = Path('/kaggle/working/bench-out')
(pub / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'Caracal Bench TPU s07', 'id': OUTPUT_DATASET,
    'licenses': [{'name': 'Apache-2.0'}]}, indent=2))
r = subprocess.run(['kaggle','datasets','create','-p',str(pub),'--public'],
                   capture_output=True, text=True, check=False)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(['kaggle','datasets','version','-p',str(pub),'-m','tpu bench'], check=False)